# 17a · Fabric: creare Eventstream (Custom endpoint) → Eventhouse
*Blocco 5 · Dati in Fabric. Passo di creazione, **prima del 17**.*

Tutti i passi per costruire in Fabric, **da codice**, il percorso che il 17 collega agli Hub:
**Custom endpoint** (sorgente) → **Eventstream** → **Eventhouse** (database KQL, tabella `Events`).

1. L'Eventhouse e il suo database KQL
2. La tabella `Events`, con le colonne del payload del device
3. L'Eventstream: sorgente *Custom endpoint*, operatore *ManageFields* con i campi del payload, destinazione *Eventhouse*
4. Controllo della topologia (stato, operatori, destinazione attiva)
5. I valori per il notebook 17b: scritti in un file (`artifacts\17a-fabric-crea-eventstream\fabric.json`), che `_start.ps1` legge da solo

**Prima:** un workspace Fabric (capacity attiva) di cui l'utente di `az login` è Admin o Member. Il workspace è quello di `WorkspaceId` in `_scenario.psd1`.
**Non ancora eseguito** su Fabric: le celle usano le API REST di Fabric (alcune in preview); se una risposta non è quella attesa, la cella lo dice e indica il passo a mano nel portale.

In [ ]:
# Carica la libreria e legge lo scenario (notebooks\_scenario.psd1): da qui in poi si usa $cfg
. ([scriptblock]::Create([IO.File]::ReadAllText((Get-ChildItem . , .\notebooks , .. -Filter _start.ps1 -ErrorAction SilentlyContinue | Select-Object -First 1).FullName)))

Scenario: rg-iot26-dev-uks-001 | namespace ADR adrns-iot-dev-uks | output in D:\Dev\2026\IoTHub\WPC2026\artifacts

## I nomi
Si cambiano qui. Il workspace esiste già: serve solo il suo id.

In [ ]:
$ws        = $cfg.Fabric.WorkspaceId
$eh        = 'eh-iot-demo'            # Eventhouse (crea anche un database KQL con lo stesso nome)
$tabella   = 'Events'
$es        = 'es-iot-demo'            # Eventstream
$api       = 'https://api.fabric.microsoft.com/v1'
$tmp       = 'D:\Dev\BuildArtifacts\tmp'   # scratch fuori dal repository
New-Item -ItemType Directory -Force $tmp | Out-Null
function Invoke-Fabric($method, $path, $body) {
    $a = @('rest','--resource','https://api.fabric.microsoft.com','--method',$method,'--url',"$api/$path",'-o','json','--only-show-errors')
    if ($body) { $f = Join-Path $tmp 'fabric-body.json'; [IO.File]::WriteAllText($f, ($body | ConvertTo-Json -Depth 20), [Text.UTF8Encoding]::new($false)); $a += @('--body',"@$f") }
    $out = az @a
    if ($LASTEXITCODE -ne 0) { throw "Fabric $method $path non riuscita" }
    if ($out) { $out | ConvertFrom-Json }
}
"Workspace $ws" 

Workspace 1bb5538f-ea39-4a44-8499-bcdb7fe0b2ef

## 1 · L'Eventhouse
Creare un Eventhouse crea anche **un database KQL con lo stesso nome**: è lì che arrivano i dati. La creazione è asincrona (risposta 202): la cella aspetta che l'item compaia.

In [ ]:
$esiste = (Invoke-Fabric get "workspaces/$ws/eventhouses").value | Where-Object displayName -eq $eh
if (-not $esiste) { Invoke-Fabric post "workspaces/$ws/eventhouses" @{ displayName = $eh } | Out-Null }
for ($i = 1; $i -le 30 -and -not $esiste; $i++) {
    Start-Sleep -Seconds 5
    $esiste = (Invoke-Fabric get "workspaces/$ws/eventhouses").value | Where-Object displayName -eq $eh
}
if (-not $esiste) { throw "Eventhouse '$eh' non creato: controlla il workspace e la capacity in Fabric." }
$kdb = (Invoke-Fabric get "workspaces/$ws/kqlDatabases").value | Where-Object displayName -eq $eh
if (-not $kdb) { throw "Il database KQL '$eh' non c'e' ancora: riesegui la cella." }
"Eventhouse $($esiste.id)  ·  database KQL $($kdb.id)  ·  query URI $($kdb.properties.queryServiceUri)" 

Eventhouse 08aaa6a5-603d-4e5b-a6c2-15bbccb24118  ·  database KQL eea1aa57-0c8d-4d84-acbe-b7765bb8c9ac  ·  query URI https://trd-yskgf2r9s06uf3b22f.z1.kusto.fabric.microsoft.com

## 2 · La tabella
Nel cluster il database si chiama con **l'id dell'item**, non con il nome. Le colonne sono quelle del **payload del device**, gli stessi campi che l'anteprima dati della Eventstream mostra: `ts`, `deviceId`, `temperature`, `setpoint`, `payloadVersion`, `alarmState`.
La Eventstream, con ingestione *ProcessedIngestion*, riempie le colonne **con lo stesso nome** dei campi in ingresso: un nome diverso resta vuoto (è successo con la prima versione, che aveva le colonne della busta CloudEvents: 16 righe tutte vuote). Se cambi il payload, guarda prima l'anteprima dati in Fabric e adegua la tabella.

In [ ]:
$q = $kdb.properties.queryServiceUri
$tok = az account get-access-token --resource $q --query accessToken -o tsv --only-show-errors
function Invoke-Kusto($endpoint, $csl) {
    (Invoke-RestMethod -Method Post -Uri "$q/v1/rest/$endpoint" -Headers @{ Authorization = "Bearer $tok" } -ContentType 'application/json; charset=utf-8' `
        -Body (@{ db = $kdb.id; csl = $csl } | ConvertTo-Json -Compress)).Tables[0]
}
$colonne = [ordered]@{ ts = 'datetime'; deviceId = 'string'; temperature = 'real'; setpoint = 'real'; payloadVersion = 'string'; alarmState = 'string' }   # i campi del payload del device
$elenco = ($colonne.GetEnumerator() | ForEach-Object { "$($_.Name):$($_.Value)" }) -join ', '
$null = Invoke-Kusto 'mgmt' ".create-merge table $tabella ($elenco)"
(Invoke-Kusto 'mgmt' ".show table $tabella cslschema").Rows | ForEach-Object { "tabella ${tabella}: " + $_[1] }

tabella Events: specversion:string,id:string,source:string,type:string,subject:string,['time']:datetime,datacontenttype:string,data:dynamic

## 3 · La Eventstream
Una definizione con quattro pezzi: la **sorgente** *Custom endpoint*, lo **stream** di default, l'operatore **ManageFields** e la destinazione *Eventhouse*.
**ManageFields serve**: con *ProcessedIngestion* la destinazione riempie solo i campi che il flusso dichiara; senza operatore la Eventstream non ha schema e la tabella si riempie di **righe vuote** (visto con 16 messaggi veri). L'operatore elenca **tutti** i campi del payload, con lo stesso nome delle colonne: uno che manca viene scartato.

In [ ]:
$definizione = [ordered]@{
    compatibilityLevel = '1.1'
    sources      = @(@{ name = 'source'; type = 'CustomEndpoint'; properties = @{} })
    streams      = @(@{ name = "$es-stream"; type = 'DefaultStream'; properties = @{}; inputNodes = @(@{ name = 'source' }) })
    destinations = @(@{
        name = 'Eventhouse'; type = 'Eventhouse'
        properties = @{
            dataIngestionMode = 'ProcessedIngestion'
            workspaceId = $ws; itemId = $kdb.id
            databaseName = $eh; tableName = $tabella
            inputSerialization = @{ type = 'Json'; properties = @{ encoding = 'UTF8' } }
        }
        inputNodes = @(@{ name = 'ManageFields' })
    })
    operators = @(@{                         # elenca i campi da portare nella tabella: gli altri vengono scartati
        name = 'ManageFields'; type = 'ManageFields'; inputNodes = @(@{ name = "$es-stream" })
        properties = @{ columns = @($colonne.Keys | ForEach-Object { @{ type = 'Rename'; alias = $_; properties = @{ column = @{ expressionType = 'ColumnReference'; node = $null; columnName = $_; columnPathSegments = @() } } } }) }
    })
}
$json = $definizione | ConvertTo-Json -Depth 20
$parte = @{ path = 'eventstream.json'; payload = [Convert]::ToBase64String([Text.Encoding]::UTF8.GetBytes($json)); payloadType = 'InlineBase64' }
$esiste = (Invoke-Fabric get "workspaces/$ws/eventstreams").value | Where-Object displayName -eq $es
if (-not $esiste) {
    Invoke-Fabric post "workspaces/$ws/eventstreams" @{ displayName = $es; definition = @{ parts = @($parte) } } | Out-Null
    for ($i = 1; $i -le 30 -and -not $esiste; $i++) { Start-Sleep -Seconds 5; $esiste = (Invoke-Fabric get "workspaces/$ws/eventstreams").value | Where-Object displayName -eq $es }
}
if (-not $esiste) { throw "Eventstream '$es' non creata." }
"Eventstream $($esiste.id)" 

Eventstream 417bd75b-4747-4eff-9d6e-d00ca5964001

## 4 · Controllo della topologia
Sorgente e destinazione devono essere **Running**. La destinazione appena creata può restare *Paused* o *Starting* per un po': se resta *Paused*, in Fabric si apre la Eventstream e si preme *Resume* sulla destinazione.

In [ ]:
$topo = Invoke-Fabric get "workspaces/$ws/eventstreams/$($esiste.id)/topology"
$topo.sources      | ForEach-Object { "sorgente     : {0} ({1}) stato {2}  id {3}" -f $_.name, $_.type, $_.status, $_.id }
"operatori    : " + @($topo.operators).Count
$topo.destinations | ForEach-Object { "destinazione : {0} ({1}) stato {2}, tabella {3}" -f $_.name, $_.type, $_.status, $_.properties.tableName }

sorgente     : source (CustomEndpoint) stato Running  id d288f1f4-12b8-4e81-a02c-d58fad3f6f73

operatori    : 0

destinazione : Eventhouse (Eventhouse) stato Creating, tabella Events

## 5 · I valori per il notebook 17b
`WorkspaceId`, `EventstreamId` e `SourceId` vengono dalla topologia. `EndpointUri` ed `EntityPath` sono nei dettagli della sorgente *Custom endpoint* (scheda *Entra ID Authentication*): la cella li legge dall'API; se non risponde, si copiano **a mano** da Fabric (Eventstream → sorgente → *Details*) nella variabile `$manuale`.
I cinque valori vengono scritti in `artifacts\17a-fabric-crea-eventstream\fabric.json` (id e nomi, **nessuna chiave**). `_start.ps1` lo legge all'avvio di ogni notebook e sovrascrive la sezione 6 di `_scenario.psd1`: il 17b e il 19 usano la Eventstream appena creata senza copiare niente.

In [ ]:
$src = $topo.sources | Where-Object type -eq 'CustomEndpoint' | Select-Object -First 1
$conn = try { Invoke-Fabric get "workspaces/$ws/eventstreams/$($esiste.id)/sources/$($src.id)/connection" } catch { $null }
$manuale = @{ EndpointUri = ''; EntityPath = '' }      # solo se l'API non risponde: namespace con sb:// davanti, e Event hub name
$valori = [ordered]@{
    WorkspaceId   = $ws
    EventstreamId = $esiste.id
    SourceId      = $src.id
    EndpointUri   = if ($conn.fullyQualifiedNamespace) { "sb://$($conn.fullyQualifiedNamespace)" } else { $manuale.EndpointUri }
    EntityPath    = if ($conn.eventHubName) { $conn.eventHubName } else { $manuale.EntityPath }
}
$vuoti = @($valori.GetEnumerator() | Where-Object { -not $_.Value })
if ($vuoti) { throw "Mancano: $(($vuoti.Name) -join ', '). Compila `$manuale con i dettagli della sorgente in Fabric e riesegui la cella." }
$cartella = Join-Path $cfg.Out '17a-fabric-crea-eventstream'
New-Item -ItemType Directory -Force $cartella | Out-Null
$file = Join-Path $cartella 'fabric.json'
$valori | ConvertTo-Json | Set-Content $file -Encoding utf8
$valori.GetEnumerator() | ForEach-Object { "{0,-14}= {1}" -f $_.Name, $_.Value }
"Scritto ${file}: il 17b e il 19 lo leggono da soli (riapri il notebook o rilancia la cella base)."


WorkspaceId   = '1bb5538f-ea39-4a44-8499-bcdb7fe0b2ef'

EventstreamId = '417bd75b-4747-4eff-9d6e-d00ca5964001'

SourceId      = 'd288f1f4-12b8-4e81-a02c-d58fad3f6f73'

EndpointUri   = 'sb://esehitnawdagz8pzmsyud4b.servicebus.windows.net'

EntityPath    = 'esehitnawdagz8pzmsyud4b_eh'

## Cosa abbiamo fatto
- Un **Eventhouse** con il suo database KQL e la tabella `Events` con le colonne del payload.
- Una **Eventstream** con sorgente *Custom endpoint* e destinazione *Eventhouse*, con l'operatore *ManageFields*.
- I cinque valori scritti in `artifacts\17a-fabric-crea-eventstream\fabric.json`, letti da `_start.ps1`.

**Da guardare dopo il 18**: l'anteprima dati della Eventstream in Fabric e il notebook 19. La Eventstream espone i campi del payload del device (visto nell'anteprima dati): se il payload cambia, si adegua la tabella, oppure si aggiunge un operatore *ManageFields* **che rinomina senza scartare gli altri campi**.

Prossimo passo: **17b**, si collegano gli Hub.